# XGBoost — enc3 × evidence 2×4 + strong-block combinations

`domain + rollup16`을 고정한 enc3/evidence 2×4와 강한 block 조합 후보를 SKF·SGKF에서 비교합니다.

- `factorial`: enc3 유무 × evidence 4수준 = 8 config, 16 case
- `candidates`: reference 4개 + 후보 8개 = 12 config, 24 case
- `all`: 중복 `f4r`을 합친 19 config, 38 case (기본값)

- 입력: `MyDrive/onco-ai-input/{train.csv,test.csv,sample_submission.csv}`
- 공용 parquet cache: `MyDrive/onco-ai-colab-results/data_process`
- 결과: `MyDrive/onco-ai-colab-results/block_ablation/xgb_enc3_evidence_all`
- 저장: case별 OOF, test probability, submission, JSON/console log, CSV/HTML 비교표
- 재개: 세 산출물과 JSON 로그가 모두 정상인 case만 건너뛰고 나머지만 실행

Colab에서 GPU 런타임(T4 이상)을 선택한 뒤 위에서부터 실행하세요.

In [ ]:
# 1) Google Drive 연결 및 경로 설정
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

REPO_URL = "https://github.com/cancer-classification-ai/onco-ai.git"
BRANCH = "test"
PROJECT_DIR = Path("/content/onco-ai")

DRIVE_ROOT = Path("/content/drive/MyDrive")
INPUT_DIR = DRIVE_ROOT / "onco-ai-input"
RESULTS_ROOT = DRIVE_ROOT / "onco-ai-colab-results"
PROCESS_DIR = RESULTS_ROOT / "data_process"
SUITE = "all"  # factorial / candidates / all
EXPECTED_CASES = {"factorial": 16, "candidates": 24, "all": 38}[SUITE]
RUN_DIR = RESULTS_ROOT / "block_ablation" / f"xgb_enc3_evidence_{SUITE}"

TAG = f"enc3_evidence_{SUITE}"
SEED = 42
RUN_DIR.mkdir(parents=True, exist_ok=True)

print("INPUT_DIR  =", INPUT_DIR)
print("PROCESS_DIR=", PROCESS_DIR)
print("RUN_DIR    =", RUN_DIR)
print("SUITE      =", SUITE, f"({EXPECTED_CASES} cases)")

In [ ]:
# 2) 저장소 clone/pull 및 XGBoost 실행에 필요한 최소 패키지 설치
import subprocess
import sys

def run(command, *, cwd=None, check=True):
    command = [str(part) for part in command]
    print("$", " ".join(command), flush=True)
    return subprocess.run(
        command,
        cwd=None if cwd is None else str(cwd),
        check=check,
    )

if (PROJECT_DIR / ".git").is_dir():
    run(["git", "fetch", "origin", BRANCH], cwd=PROJECT_DIR)
    run(["git", "checkout", BRANCH], cwd=PROJECT_DIR)
    run(["git", "pull", "--ff-only", "origin", BRANCH], cwd=PROJECT_DIR)
else:
    run([
        "git", "clone", "--branch", BRANCH, "--single-branch",
        REPO_URL, PROJECT_DIR,
    ])

run([
    sys.executable, "-m", "pip", "install", "-q",
    "numpy", "pandas", "scipy", "scikit-learn", "pyarrow", "xgboost>=2,<4",
])

COMMIT_SHA = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=PROJECT_DIR, text=True
).strip()
print("Git commit =", COMMIT_SHA)

In [ ]:
# 3) 입력 파일과 GPU 상태 확인
import pandas as pd
import xgboost as xgb

REQUIRED_INPUTS = ("train.csv", "test.csv", "sample_submission.csv")
missing = [name for name in REQUIRED_INPUTS if not (INPUT_DIR / name).is_file()]
assert not missing, f"입력 파일이 없습니다: {missing}"

train_columns = pd.read_csv(INPUT_DIR / "train.csv", nrows=0).columns
test_columns = pd.read_csv(INPUT_DIR / "test.csv", nrows=0).columns
assert {"ID", "SUBCLASS"}.issubset(train_columns)
assert "ID" in test_columns and "SUBCLASS" not in test_columns
assert list(train_columns.drop("SUBCLASS")) == list(test_columns)

run(["nvidia-smi"], check=False)
print("XGBoost version =", xgb.__version__)
print("CUDA build      =", xgb.build_info().get("USE_CUDA", False))
print("train columns   =", len(train_columns), "(ID + SUBCLASS + genes)")
print("test columns    =", len(test_columns), "(ID + genes)")

In [ ]:
# 4) 누락 parquet만 생성하고 선택한 suite를 SKF/SGKF로 실행
# 런타임이 끊기면 이 셀을 다시 실행하세요. 완료된 case는 자동으로 건너뜁니다.
command = [
    sys.executable,
    "-u",  # stdout unbuffered: START/fold/DONE 로그를 Colab에서 즉시 표시
    PROJECT_DIR / "scripts" / "run_xgb_feature_combinations.py",
    "--input-dir", INPUT_DIR,
    "--process-dir", PROCESS_DIR,
    "--output-dir", RUN_DIR,
    "--device", "gpu",
    "--suite", SUITE,
    "--seed", str(SEED),
    "--tag", TAG,
]
run(command, cwd=PROJECT_DIR)

In [ ]:
# 5) 최종 비교표: 전체 순위 + 2×4 표 + 후보/anchor delta
from IPython.display import display

comparison_path = RUN_DIR / "reports" / f"comparison_{TAG}_s{SEED}.csv"
comparison = pd.read_csv(comparison_path)

ranking_columns = [
    "config", "cv", "group", "enc3", "evidence", "n_features",
    "oof_macro_f1", "oof_macro_f1_singleton", "oof_accuracy",
    "anchor_config", "delta_vs_anchor",
    "delta_evidence_vs_none", "delta_enc3_same_evidence", "device",
]
print("전체 순위")
display(
    comparison[ranking_columns]
    .sort_values(["cv", "oof_macro_f1"], ascending=[True, False])
    .style.format({
        "oof_macro_f1": "{:.6f}",
        "oof_macro_f1_singleton": "{:.6f}",
        "oof_accuracy": "{:.6f}",
        "delta_vs_anchor": "{:+.6f}",
        "delta_evidence_vs_none": "{:+.6f}",
        "delta_enc3_same_evidence": "{:+.6f}",
    })
)

candidate_rows = comparison[comparison["group"].isin(["reference", "candidate"])]
if not candidate_rows.empty:
    print("\n강한 block 조합 후보 — 같은 CV anchor 대비 delta")
    display(
        candidate_rows[[
            "config", "cv", "group", "anchor_config", "n_features",
            "oof_macro_f1", "oof_macro_f1_singleton", "delta_vs_anchor",
        ]]
        .sort_values(["cv", "oof_macro_f1"], ascending=[True, False])
        .style.format({
            "oof_macro_f1": "{:.6f}",
            "oof_macro_f1_singleton": "{:.6f}",
            "delta_vs_anchor": "{:+.6f}",
        })
    )

for cv in ("skf", "sgkf"):
    subset = comparison[(comparison["cv"] == cv) & (comparison["group"] == "factorial")].copy()
    if subset.empty:
        continue
    score_table = subset.pivot(index="evidence", columns="enc3", values="oof_macro_f1")
    score_table = score_table.rename(columns={False: "enc3 없음", True: "enc3 있음"})
    score_table["enc3 효과"] = score_table["enc3 있음"] - score_table["enc3 없음"]
    score_table = score_table.reindex(["none", "ebovr", "ebbnb", "both"])
    print(f"\n{cv.upper()} 2×4 Macro F1")
    display(score_table.style.format("{:.6f}"))

print("\n비교표 CSV :", comparison_path)
print("비교표 HTML:", comparison_path.with_suffix(".html"))
print("OOF         :", RUN_DIR / "artifacts" / "oof")
print("test proba  :", RUN_DIR / "artifacts" / "test_predictions")
print("submission  :", RUN_DIR / "artifacts" / "submissions")
print("JSON logs   :", RUN_DIR / "artifacts" / "logs")

assert len(comparison) == EXPECTED_CASES, f"완료된 case가 {len(comparison)}/{EXPECTED_CASES}개입니다. 4번 셀을 다시 실행하세요."